# 第26章　損失関数を実装する ― 医療タスクに合わせて設計する

**『医療診断支援AI開発　基礎編 ― 自分で作る（基礎編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-basic

## 26.1　クラス重み付き交差エントロピー

In [ ]:
import torch.nn as nn
import torch, torch.nn as nn
weights = torch.tensor([1.0, 5.0]).to(device)   # 背景1.0 / 病変5.0
criterion = nn.CrossEntropyLoss(weight=weights)

## 26.2　Dice損失 ― 重なりを直接最適化する

In [ ]:
import torch.nn as nn

def dice_loss(logits, target, eps=1e-6):
    # target は (B, H, W) の整数ラベル。(B, 1, H, W) 形式なら target.squeeze(1) を通してから渡す
    prob = logits.softmax(1)                     # (B, C, ...)
    target_1h = nn.functional.one_hot(target, prob.size(1)).movedim(-1, 1).float()
    dims = tuple(range(2, prob.ndim))            # 空間次元で集計
    inter = (prob * target_1h).sum(dims)
    union = prob.sum(dims) + target_1h.sum(dims)
    return 1 - ((2*inter + eps) / (union + eps)).mean()

## 26.3　Tversky損失 ― 見逃しを重く罰する

In [ ]:
def tversky_loss(logits, target, alpha=0.3, beta=0.7, eps=1e-6):
    prob = logits.softmax(1)[:, 1]               # 病変クラスの確率 (B,H,W)
    if target.dim() == prob.dim() + 1:           # MONAI流の (B,1,H,W) が来たら
        target = target.squeeze(1)               # チャネル次元を外す
    # 形を確かめてから掛ける。(B,H,W)×(B,1,H,W) は (B,B,H,W) へ broadcast され、
    # 症例をまたいだ組合せまで足し込んでしまう（B=2 で 4 通り）。
    assert prob.shape == target.shape, (prob.shape, target.shape)
    t = (target == 1).float()
    tp = (prob * t).sum()
    fp = (prob * (1 - t)).sum()                  # 過検出
    fn = ((1 - prob) * t).sum()                  # 見逃し（beta大で重く罰す）
    return 1 - (tp + eps) / (tp + alpha*fp + beta*fn + eps)

## 26.4　組み合わせて使う

In [ ]:
from monai.losses import DiceCELoss, TverskyLoss
criterion = DiceCELoss(to_onehot_y=True, softmax=True)          # Dice + 交差エントロピー
# criterion = TverskyLoss(to_onehot_y=True, softmax=True, alpha=0.3, beta=0.7)

## よくある誤解とミス ― 損失関数で静かに事故る

In [ ]:
import torch.nn as nn

logits = model(x)                     # (B, C) 生のロジット。softmaxは通さない
loss = nn.CrossEntropyLoss()(logits, target)   # 内部でlog_softmaxされる

## 損失の数字を、手で一度追う ― CEとDiceが返す値の意味

In [ ]:
import torch.nn.functional as F
import torch, torch.nn.functional as F
for z in [(0., 4.), (0., 0.), (4., 0.)]:                 # 正解は常にクラス1
    logits = torch.tensor([z])
    ce = F.cross_entropy(logits, torch.tensor([1]))
    print(z, f"softmax_1={logits.softmax(1)[0,1]:.3f}  CE={ce.item():.3f}")
# (0., 4.) softmax_1=0.982  CE=0.018   自信あり・正解 → ほぼ0
# (0., 0.) softmax_1=0.500  CE=0.693   五分五分       → ln2 = 0.693
# (4., 0.) softmax_1=0.018  CE=4.018   自信あり・不正解 → 重く罰される

In [ ]:
g = torch.tensor([1., 1., 0., 0.])                        # 正解マスク
for p_lesion in [0.5, 0.7, 0.9]:
    p = torch.tensor([p_lesion, p_lesion, 1 - p_lesion, 1 - p_lesion])
    soft_dice = (2 * (p * g).sum()) / (p.sum() + g.sum())
    print(f"病変画素の確率={p_lesion}: soft-Dice={soft_dice:.3f}")
# 0.5→0.500, 0.7→0.700, 0.9→0.900 と、確率に応じて滑らかに動く